In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from datetime import datetime

# 1. Load the dataset
try:
    df = pd.read_csv('product_price_dataset_multiloc.csv')
except FileNotFoundError:
    print("Error: CSV file not found.")
    exit()


# 2. Season Detection Logic
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4]:
        return 'Summer'
    elif month in [5, 6, 7, 8]:
        return 'Rainy'
    elif month in [9, 10, 11]:
        return 'Autumn'
    return 'Winter'


# 3. Preprocessing
def preprocess_data(df):
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.dayofweek

    encoders = {}
    cat_cols = ['product', 'location', 'season']
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le
    return df, encoders


df_processed, encoders = preprocess_data(df)

# Define features and target
features = ['product', 'location', 'season', 'month', 'year', 'day_of_week']
X = df_processed[features]
y = df_processed['product_price_per_unit']

# --- STEP 4: TRAIN/TEST SPLIT ---
# We reserve 20% of the data to test how well the model predicts unseen data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- STEP 5: TRAINING ---
model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    early_stopping_rounds=50,  # Prevents overfitting
    objective='reg:squarederror'
)

# Fit model with evaluation set to monitor performance
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# --- STEP 6: ACCURACY EVALUATION ---
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print("MODEL PERFORMANCE METRICS")
print(f"R-squared Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f} (Average error in price)")
print("-" * 30)


# 7. Prediction Interface
def predict_price():
    print("\n--- Future Price Predictor ---")
    try:
        u_loc = input(f"Location {list(encoders['location'].classes_)}: ").strip()
        u_prod = input(f"Product {list(encoders['product'].classes_)}: ").strip()
        u_date_str = input("Enter Future Date (YYYY-MM-DD): ").strip()

        target_date = datetime.strptime(u_date_str, '%Y-%m-%d')
        detected_season = get_season(target_date)

        # Encode inputs
        input_features = pd.DataFrame([{
            'product': encoders['product'].transform([u_prod])[0],
            'location': encoders['location'].transform([u_loc])[0],
            'season': encoders['season'].transform([detected_season])[0],
            'month': target_date.month,
            'year': target_date.year,
            'day_of_week': target_date.weekday()
        }])

        prediction = model.predict(input_features[features])[0]

        print(f"\nResult: Predicted Price is ${prediction:.2f} (Season: {detected_season})")

    except Exception as e:
        print(f"\nError: {e}")


if __name__ == "__main__":
    predict_price()


------------------------------
MODEL PERFORMANCE METRICS
R-squared Score: 0.9987
RMSE: 0.9459 (Average error in price)
------------------------------

--- Future Price Predictor ---

Result: Predicted Price is $59.20 (Season: Summer)
